In [1]:
# %% [0] Phase 1 -- layered-medium Green's function for the TFLN stack
#
# Builds and validates the mixed-potential Green's function that is the kernel
# of the layered-medium MoM (Route C).  Same-interface scalar-potential G_q and
# vector-potential G_A on the metal plane, inverted from the transmission-line
# spectral kernels by a Sommerfeld integral with quasi-static singularity
# extraction and a Hankel-split deformed contour (real head past the surface-
# wave poles, then two exponentially-decaying rays).
#
# Deliverables written by this notebook:
#   green_function_table.npz   the validated 1D table G_q(rho), G_A(rho)
#   te0_pole.json              the TE0 (and TM0) surface-wave pole
#   phase1_greens.png          the 3-panel figure
#   phase1_report.md           the one-page report
#
# The Green's function machinery lives in layered_greens.py (reused by later
# phases); the stack constants live in stack_params.py.
import json, time, warnings
import numpy as np
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from layered_greens import Stack, C0, EPS0, MU0
import stack_params as sp

f = sp.F0
k0 = 2 * np.pi * f / C0
print(f"stack: air | LN {sp.TFLN*1e6:.2f} um (eps {sp.EPS_LN}) | "
      f"SiO2 {sp.BOX_H*1e6:.1f} um (eps {sp.EPS_SIO2}) | "
      f"Si {sp.SI_H*1e6:.0f} um (eps {sp.EPS_SI}) | air     f = {f/1e9:.0f} GHz")


stack: air | LN 0.46 um (eps 34.7) | SiO2 4.7 um (eps 3.9) | Si 550 um (eps 11.7) | air     f = 60 GHz


In [2]:
# %% [1] Build and save the Green's function table (device stack)
# ============================================================
DEV = Stack(sp.EPS_AIR, sp.device_layers(lossy=True), sp.EPS_AIR, f=f)
rho = np.logspace(np.log10(1e-8), np.log10(1e-2), 90)     # 10 nm ... 10 mm
t0 = time.time()
G_A, G_q = DEV.table(rho)
t_table = time.time() - t0
np.savez("green_function_table.npz", rho=rho, G_A=G_A, G_q=G_q,
         f=f, layers=np.array([(e.real, e.imag, d) for e, d in sp.device_layers()]),
         top_eps=sp.EPS_AIR, bot_eps=sp.EPS_AIR)


In [3]:
# %% [2] Surface-wave poles
# ============================================================
DEV_LL = Stack(sp.EPS_AIR, sp.device_layers(lossy=False), sp.EPS_AIR, f=f)
te = DEV_LL.surface_waves("TE")
tm = DEV_LL.surface_waves("TM")
n_te0 = te[0]
resid = abs(DEV_LL._sw_det(n_te0, "TE"))
json.dump({"n_TE0": n_te0, "residual": resid, "TM_poles": tm},
          open("te0_pole.json", "w"), indent=2)


In [4]:
# %% [3] Validation gate V1.1 - V1.6
# ============================================================
res = {}

# V1.1 free space
S = Stack(1.0, [], 1.0, f=f)
e11 = max(abs(S.Gq(r) - np.exp(-1j*k0*r)/(4*np.pi*EPS0*r)) /
          abs(np.exp(-1j*k0*r)/(4*np.pi*EPS0*r)) for r in (1e-6, 1e-5, 1e-4))
res["V1.1"] = ("free-space limit", e11, 1e-4, e11 < 1e-4)

# V1.2 PEC image (vacuum layer h over PEC; image charge -q at depth 2h)
h = 5e-6
SP = Stack(1.0, [(1.0, h)], 1e7, f=f)
def _img(r):
    R = np.sqrt(r**2 + (2*h)**2)
    return (np.exp(-1j*k0*r)/r - np.exp(-1j*k0*R)/R) / (4*np.pi*EPS0)
e12 = max(abs(SP.Gq(r) - _img(r)) / abs(_img(r)) for r in (1e-6, 5e-6, 2e-5))
res["V1.2"] = ("PEC-ground image", e12, 1e-3, e12 < 1e-3)

# V1.3 single dielectric interface eps2 = 3.9 (static two-dielectric image)
S2 = Stack(1.0, [], 3.9, f=f)
e13 = max(abs(S2.Gq(r).real - 1/(2*np.pi*EPS0*(1+3.9)*r)) /
          (1/(2*np.pi*EPS0*(1+3.9)*r)) for r in (1e-7, 1e-6, 1e-5))
res["V1.3"] = ("single interface eps=3.9", e13, 1e-2, e13 < 1e-2)

# V1.4 TE0 pole
e14 = abs(n_te0 - sp.N_TE0_REF)
res["V1.4"] = ("TE0 pole (=2.5414)", e14, 1e-3, e14 < 1e-3)

# V1.5 reciprocity / kernel is a function of rho=|r-r'| only (symmetry)
# G(r,r') depends only on |rho|; evaluate at rho and -rho (same |rho|).
r0 = 3e-6
e15 = abs(DEV.Gq(r0) - DEV.Gq(abs(-r0))) / abs(DEV.Gq(r0))
res["V1.5"] = ("reciprocity / rho-symmetry", e15, 1e-12, e15 < 1e-12)

# V1.6 interpolation consistency: log-log interpolate the table, compare to a
# fresh direct evaluation at intermediate rho.
from scipy.interpolate import CubicSpline
lr = np.log(rho)
csr = CubicSpline(lr, G_q.real); csi = CubicSpline(lr, G_q.imag)
rho_mid = np.sqrt(rho[20:70:8] * rho[21:71:8])          # intermediate field, off-grid
def direct(rm):
    return DEV._sommerfeld(rm, DEV.Gq_spectral, DEV._asym("TM"))
e16 = max(abs((csr(np.log(rm)) + 1j*csi(np.log(rm))) - direct(rm)) / abs(direct(rm))
          for rm in rho_mid)
res["V1.6"] = ("table interpolation", e16, 1e-3, e16 < 1e-3)

print("="*68)
print("PHASE 1 VALIDATION GATE")
print("-"*68)
allpass = True
for k in sorted(res):
    name, val, thr, ok = res[k]
    allpass &= ok
    print(f"  {k}  {name:<28} {val:.2e}  (< {thr:.0e})  {'PASS' if ok else 'FAIL'}")
print("-"*68)
print(f"  TE0 pole n = {n_te0:.5f}  (residual {resid:.1e})   TM pole n = {tm[0]:.5f}")
print(f"  table: {len(rho)} rho points built in {t_table:.1f} s")
print(f"  {'ALL PASS' if allpass else 'SOME FAILED'}")
print("="*68)


PHASE 1 VALIDATION GATE
--------------------------------------------------------------------
  V1.1  free-space limit             8.41e-14  (< 1e-04)  PASS
  V1.2  PEC-ground image             8.66e-08  (< 1e-03)  PASS
  V1.3  single interface eps=3.9     2.62e-04  (< 1e-02)  PASS
  V1.4  TE0 pole (=2.5414)           1.00e-05  (< 1e-03)  PASS
  V1.5  reciprocity / rho-symmetry   0.00e+00  (< 1e-12)  PASS
  V1.6  table interpolation          4.75e-05  (< 1e-03)  PASS
--------------------------------------------------------------------
  TE0 pole n = 2.54141  (residual 1.6e-14)   TM pole n = 1.15066
  table: 90 rho points built in 17.0 s
  ALL PASS


In [5]:
# %% [4] Figure and report
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].loglog(rho*1e6, np.abs(G_q), color="#2c7fb8")
ax[0].set_xlabel(r"$\rho$ ($\mu$m)"); ax[0].set_ylabel(r"$|G_q|$ (V m / C)")
ax[0].set_title(r"(a) scalar-potential kernel $G_q(\rho)$")
ax[1].loglog(rho*1e6, np.abs(G_A), color="#e67e22")
ax[1].set_xlabel(r"$\rho$ ($\mu$m)"); ax[1].set_ylabel(r"$|G_A|$ (H/m)")
ax[1].set_title(r"(b) vector-potential kernel $G_A(\rho)$")
# (c) spectral integrand magnitude vs krho, showing the TM & TE poles
kr = np.linspace(1.001*k0, sp.N_TE0_REF*k0*1.05, 4000)
gt = np.abs([DEV_LL.Gq_spectral(k) for k in kr])
ht = np.abs([DEV_LL.GA_spectral(k) for k in kr])
ax[2].semilogy(kr/k0, gt/gt.max(), color="#2c7fb8", label=r"$\tilde G_q$ (TM)")
ax[2].semilogy(kr/k0, ht/ht.max(), color="#e67e22", label=r"$\tilde G_A$ (TE)")
ax[2].axvline(tm[0], color="#2c7fb8", ls=":", lw=1)
ax[2].axvline(n_te0, color="#e67e22", ls=":", lw=1)
ax[2].set_xlabel(r"$k_\rho / k_0$  (= effective index)")
ax[2].set_ylabel("normalised spectral kernel")
ax[2].set_title(f"(c) surface-wave poles: TM {tm[0]:.3f}, TE {n_te0:.3f}")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("phase1_greens.png", dpi=130); plt.close()

# ============================================================
# report
# ============================================================
with open("phase1_report.md", "w") as fh:
    fh.write("# Phase 1 report -- layered-medium Green's function\n\n")
    fh.write(f"Stack: air | LiNbO3 {sp.TFLN*1e6:.2f} um (eps {sp.EPS_LN}) | "
             f"SiO2 {sp.BOX_H*1e6:.1f} um (eps {sp.EPS_SIO2}) | "
             f"Si {sp.SI_H*1e6:.0f} um (eps {sp.EPS_SI}, sigma {sp.SIGMA_SI:g}) | air, "
             f"at {f/1e9:.0f} GHz.\n\n")
    fh.write(f"**TE0 surface-wave pole: n = {n_te0:.5f}** (residual {resid:.1e}); "
             f"TM0 pole n = {tm[0]:.5f}.\n\n")
    fh.write("## Validation gate\n\n| test | what | error | threshold | result |\n")
    fh.write("|---|---|---|---|---|\n")
    for k in sorted(res):
        name, val, thr, ok = res[k]
        fh.write(f"| {k} | {name} | {val:.2e} | {thr:.0e} | "
                 f"{'PASS' if ok else 'FAIL'} |\n")
    fh.write(f"\nAll six {'PASS' if allpass else 'did NOT all pass'}. "
             f"Table: {len(rho)} log-spaced rho points (10 nm - 10 mm), "
             f"built in {t_table:.1f} s.\n\n")
    fh.write("Figure: `phase1_greens.png` -- (a) |G_q|, (b) |G_A|, "
             "(c) spectral kernels with the TM and TE surface-wave poles.\n\n")
    fh.write(f"G_q at rho=1 um: {G_q[np.argmin(abs(rho-1e-6))]:.4e};  "
             f"at rho=100 um: {G_q[np.argmin(abs(rho-1e-4))]:.4e}.\n")
print("wrote green_function_table.npz, te0_pole.json, phase1_greens.png, phase1_report.md")


wrote green_function_table.npz, te0_pole.json, phase1_greens.png, phase1_report.md
